# PetenFire — Entrenamiento M1 TEMPORADA (Mar-May) en Google Colab

Solo meses 3, 4, 5 — temporada crítica de incendios del Petén

**Base rate ~0.176% train / 0.262% val / 0.288% test** vs 0.064% anual | **Subsampleo 5:1** (negativos:positivos)

**Dataset:** `m1_dataset.parquet` | **Modelo:** LightGBM (probabilidades RAW, sin calibración isotónica)

---

## Instrucciones previas

1. Sube `m1_dataset.parquet` a Google Drive en la ruta: `Mi unidad/petenfire/data/`
2. Activa GPU: `Entorno de ejecución → Cambiar tipo de entorno → GPU T4 (gratis)`
3. Ejecuta **Run All** (`Ctrl+F9`)

---

### Estadísticas de temporada (meses 3, 4, 5)
| Split | Años | Filas (temporada) | Fuegos | Base rate |
|-------|------|-------------------|--------|----------|
| train | 2018–2022 | 24,477,520 | 42,979 | **0.176%** |
| val   | 2023      | 4,895,504  | 12,804 | **0.262%** |
| test  | 2024      | 4,895,504  | 14,079 | **0.288%** |

> Comparar con base rate anual de 0.064% — restricción a temporada mejora la señal ~3–4×.

### Criterio de éxito
| Métrica | Mínimo |
|---------|--------|
| AUC-PR (val) | > 0.10 |
| F1 (val, umbral óptimo) | > 0.20 |
| best_iteration_ | > 1 |

> **Nota:** El umbral de decisión se determina automáticamente buscando el máximo F1
> en la curva precision-recall de val (Celda 8). El valor se guarda en el artefacto
> y es el que debe usarse en inferencia **solo durante marzo–mayo**.

In [ ]:
# Celda 2 — Instalar dependencias
# lightgbm, scikit-learn y joblib suelen venir en Colab; forzamos versiones estables
!pip install -q lightgbm scikit-learn joblib pandas pyarrow numpy psutil
print("Dependencias instaladas.")

In [ ]:
# Celda 3 — Montar Google Drive
import os

from google.colab import drive

drive.mount("/content/drive", force_remount=True)

# Ajusta estas rutas si guardaste el archivo en otra carpeta
DATASET_PATH = "/content/drive/MyDrive/petenfire/data/m1_dataset.parquet"
OUTPUT_DIR = "/content/drive/MyDrive/petenfire/models/"
MODEL_FILENAME = "m1_lightgbm_season.joblib"  # nombre distinto al modelo anual

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Verificar que el dataset existe antes de continuar
if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(
        f"No se encontró el dataset en: {DATASET_PATH}\n"
        "Asegúrate de haberlo subido a Google Drive en la carpeta "
        "'Mi unidad/petenfire/data/' y de haber montado el Drive correctamente."
    )

print(f"Dataset encontrado: {DATASET_PATH}")
print(f"Directorio de salida: {OUTPUT_DIR}")
print(f"Archivo de modelo: {MODEL_FILENAME}")

In [ ]:
# Celda 4 — Cargar datos con filtro de TEMPORADA y subsampleo RAM-eficiente
#
# Estrategia:
#   - Leer el parquet en batches de 5 M filas para evitar picos de RAM
#   - Filtrar SOLO meses 3, 4, 5 (temporada crítica de incendios del Petén)
#   - train: todos los positivos + muestra 5:1 de negativos (seed=42)
#   - val / test: subsampleo a máx 500 K filas (proporcional, estratificado)
#
# Estadísticas de temporada conocidas:
#   train: 24,477,520 filas | 42,979 fuegos | 0.176%
#   val:    4,895,504 filas | 12,804 fuegos | 0.262%
#   test:   4,895,504 filas | 14,079 fuegos | 0.288%

import gc

import numpy as np
import pandas as pd
import psutil
import pyarrow.parquet as pq

RANDOM_SEED = 42
SEASON_MONTHS = [3, 4, 5]          # marzo, abril, mayo — temporada crítica
NEG_RATIO = 5                      # 5 negativos por positivo (antes era 10 en modelo anual)
VAL_TEST_MAX_ROWS = 500_000        # máximo de filas para val y test
BATCH_SIZE = 5_000_000             # filas por batch de lectura

FEATURE_COLS = [
    # Clima
    "T2M", "RH2M", "WS10M", "PRECTOTCORR",
    # FWI
    "fwi", "ffmc_val", "dmc_val", "dc_val", "isi_val", "bui_val",
    # Precipitación acumulada
    "prec_acc7d", "prec_acc14d",
    # Vegetación
    "ndvi", "ndvi_lag7", "ndvi_lag14",
    # Topografía
    "elevation_m", "slope_deg", "aspect_deg",
    # Antrópico
    "dist_roads_km", "dist_settlements_km", "is_protected_area",
    # Temporal
    "month", "day_of_year",
]
TARGET_COL = "fire_occurred"
COLS_NEEDED = FEATURE_COLS + [TARGET_COL, "split", "month"]


def ram_gb() -> float:
    """RAM usada por el proceso actual en GB."""
    return psutil.Process().memory_info().rss / 1e9


def subsample_split_season(
    pf: pq.ParquetFile,
    split_name: str,
    season_months: list[int],
    ratio: int = 5,
    max_rows: int | None = None,
    seed: int = 42,
) -> pd.DataFrame:
    """Lee el parquet en batches, filtra la temporada y retorna un DataFrame subsampled.

    Args:
        pf: ParquetFile abierto.
        split_name: 'train', 'val' o 'test'.
        season_months: Lista de meses a incluir (ej: [3, 4, 5]).
        ratio: Negativos por positivo en train. Ignorado para val/test.
        max_rows: Máximo de filas para val/test. None = sin límite.
        seed: Semilla aleatoria.

    Returns:
        DataFrame con datos del split filtrado por temporada y subsampled.
    """
    rng = np.random.default_rng(seed)
    pos_list: list[pd.DataFrame] = []
    neg_list: list[pd.DataFrame] = []

    print(f"  Leyendo split '{split_name}' en batches de {BATCH_SIZE:,} filas...")
    for batch in pf.iter_batches(batch_size=BATCH_SIZE, columns=list(set(COLS_NEEDED))):
        df_batch = batch.to_pandas()

        # Filtro 1: split correcto
        df_batch = df_batch[df_batch["split"] == split_name]
        if df_batch.empty:
            continue

        # Filtro 2: solo meses de temporada (ANTES del subsampleo)
        df_batch = df_batch[df_batch["month"].isin(season_months)]
        if df_batch.empty:
            continue

        pos_batch = df_batch[df_batch[TARGET_COL] == 1]
        neg_batch = df_batch[df_batch[TARGET_COL] == 0]

        pos_list.append(pos_batch)

        if split_name == "train":
            # Subsampleo 5:1 sobre datos ya filtrados por temporada
            n_neg_keep = min(len(neg_batch), len(pos_batch) * ratio)
            if n_neg_keep > 0:
                idx = rng.choice(len(neg_batch), size=n_neg_keep, replace=False)
                neg_list.append(neg_batch.iloc[idx])
        else:
            neg_list.append(neg_batch)

    all_pos = pd.concat(pos_list, ignore_index=True) if pos_list else pd.DataFrame()
    all_neg = pd.concat(neg_list, ignore_index=True) if neg_list else pd.DataFrame()

    if split_name != "train" and max_rows is not None:
        # Para val/test: subsamplear al total deseado manteniendo el ratio original
        total = len(all_pos) + len(all_neg)
        if total > max_rows:
            frac = max_rows / total
            n_pos = min(len(all_pos), max(1, int(len(all_pos) * frac)))
            n_neg = max_rows - n_pos
            idx_pos = rng.choice(len(all_pos), size=n_pos, replace=False)
            idx_neg = rng.choice(len(all_neg), size=min(n_neg, len(all_neg)), replace=False)
            all_pos = all_pos.iloc[idx_pos]
            all_neg = all_neg.iloc[idx_neg]

    result = pd.concat([all_pos, all_neg], ignore_index=True)
    result = result.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return result


# Verificar columnas disponibles antes de cargar todo
pf = pq.ParquetFile(DATASET_PATH)
schema_names = set(pf.schema_arrow.names)
available_features = [c for c in FEATURE_COLS if c in schema_names]
missing_features = [c for c in FEATURE_COLS if c not in schema_names]

# Verificar que 'month' existe (necesario para el filtro de temporada)
if "month" not in schema_names:
    raise ValueError(
        "La columna 'month' no existe en el parquet. "
        "No es posible filtrar por temporada sin esta columna."
    )

if missing_features:
    print(f"AVISO: Features no encontradas en el parquet (se omitirán): {missing_features}")
print(f"Features disponibles: {len(available_features)}/{len(FEATURE_COLS)}")
print(f"Filtro de temporada: meses {SEASON_MONTHS} (mar, abr, may)")
print(f"Ratio de subsampleo en train: {NEG_RATIO}:1 (negativos:positivos)")

# Ajustar COLS_NEEDED con solo las columnas que existen
COLS_NEEDED = available_features + [TARGET_COL, "split", "month"]

print(f"\nRAM antes de carga: {ram_gb():.2f} GB")
print("Cargando train (temporada)...")
train_df = subsample_split_season(
    pf, "train", season_months=SEASON_MONTHS, ratio=NEG_RATIO, seed=RANDOM_SEED
)
print(f"  train: {len(train_df):,} filas | positivos: {train_df[TARGET_COL].mean()*100:.2f}%")

print("Cargando val (temporada)...")
val_df = subsample_split_season(
    pf, "val", season_months=SEASON_MONTHS, max_rows=VAL_TEST_MAX_ROWS, seed=RANDOM_SEED
)
print(f"  val:   {len(val_df):,} filas | positivos: {val_df[TARGET_COL].mean()*100:.4f}%")

print("Cargando test (temporada)...")
test_df = subsample_split_season(
    pf, "test", season_months=SEASON_MONTHS, max_rows=VAL_TEST_MAX_ROWS, seed=RANDOM_SEED
)
print(f"  test:  {len(test_df):,} filas | positivos: {test_df[TARGET_COL].mean()*100:.4f}%")

print(f"\nRAM después de carga: {ram_gb():.2f} GB")
gc.collect()

In [ ]:
# Celda 5 — Preparar arrays numpy

import numpy as np

# Excluir 'month' de las features si no está en FEATURE_COLS originales
# (puede haberse colado en COLS_NEEDED para el filtro; solo usar FEATURE_COLS)
X_train = train_df[available_features].values.astype(np.float32)
y_train = train_df[TARGET_COL].values.astype(np.int32)

X_val = val_df[available_features].values.astype(np.float32)
y_val = val_df[TARGET_COL].values.astype(np.int32)

X_test = test_df[available_features].values.astype(np.float32)
y_test = test_df[TARGET_COL].values.astype(np.int32)

# Liberar DataFrames para recuperar RAM
del train_df, val_df, test_df
gc.collect()

print("Shapes y ratios de positivos:")
for name, X, y in [("train", X_train, y_train), ("val", X_val, y_val), ("test", X_test, y_test)]:
    print(f"  {name:6s}: X={X.shape} | y positivos={y.sum():,} ({y.mean()*100:.3f}%)")

print(f"\nRAM después de conversión a numpy: {ram_gb():.2f} GB")

# Diagnóstico de NaN (útil si el modelo no aprende)
nan_frac = np.isnan(X_train).mean(axis=0)
high_nan = [(available_features[i], f"{nan_frac[i]*100:.1f}%")
            for i in range(len(available_features)) if nan_frac[i] > 0.1]
if high_nan:
    print(f"\nAVISO — Features con >10% NaN en train: {high_nan}")
else:
    print("\nOK — Ninguna feature supera 10% de NaN en train.")

In [ ]:
# Celda 6 — Entrenar LightGBM (modelo de TEMPORADA)
#
# Diferencias respecto al modelo anual:
#   - num_leaves=63 (más profundo, datos más informativos por ser solo temporada)
#   - max_depth=6 (límite explícito para evitar overfitting)
#   - scale_pos_weight=1.0 (subsampleo 5:1 ya equilibró clases)
#   - eval_metric='average_precision' (AUC-PR, apropiado para clases raras)
#   - early_stopping(30): para si no mejora en 30 rondas consecutivas

import time

from lightgbm import LGBMClassifier, early_stopping, log_evaluation

N_ESTIMATORS = 500
LEARNING_RATE = 0.05
MAX_DEPTH = 6           # más profundo que el anual (sin límite) para capturar patrones de temporada
NUM_LEAVES = 63         # 2^MAX_DEPTH - 1; equilibrio entre expresividad y regularización

lgbm_model = LGBMClassifier(
    n_estimators=N_ESTIMATORS,
    max_depth=MAX_DEPTH,
    learning_rate=LEARNING_RATE,
    scale_pos_weight=1.0,   # subsampleo 5:1 ya equilibró el dataset
    num_leaves=NUM_LEAVES,  # controla complejidad; 63 = buen punto de partida para depth=6
    min_child_samples=20,   # evita hojas con muy pocas muestras
    random_state=RANDOM_SEED,
    n_jobs=-1,
    verbose=-1,
)

print("Iniciando entrenamiento LightGBM (modelo de temporada mar-may)...")
t0 = time.time()

lgbm_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="average_precision",
    feature_name=available_features,
    callbacks=[
        early_stopping(30, verbose=False),
        log_evaluation(50),
    ],
)

elapsed = time.time() - t0
print(f"\nEntrenamiento completado en {elapsed/60:.1f} min")
print(f"Mejor iteración: {lgbm_model.best_iteration_}")

if lgbm_model.best_iteration_ <= 1:
    print("ADVERTENCIA: best_iteration_ = 1. El modelo probablemente no aprendió.")
    print("  → Verifica que el dataset tenga positivos y que las features no sean todas NaN.")
    print("  → Ejecuta la celda 5 y revisa la sección de diagnóstico de NaN.")

In [ ]:
# Celda 7 — Probabilidades RAW del modelo (sin calibración isotónica)
#
# La calibración isotónica entrenada sobre val COMPLETO (prevalencia ~0.064%)
# aprende a mapear scores de ~9% → ~0.064%, comprimiendo todo por debajo de
# cualquier umbral fijo útil. Resultado: F1=0 aunque AUC-ROC sea excelente.
#
# Con el modelo de temporada la prevalencia es ~0.262% en val — mucho mejor,
# pero seguimos usando scores RAW para maximizar el rango dinámico útil.

import numpy as np
import pandas as pd

# Pasar DataFrames con nombres de columnas para evitar el warning
# "X does not have valid feature names" (modelo entrenado con feature_name=)
X_val_df  = pd.DataFrame(X_val,  columns=available_features)
X_test_df = pd.DataFrame(X_test, columns=available_features)

# Probabilidades RAW directamente del LightGBM
raw_proba_val  = lgbm_model.predict_proba(X_val_df)[:, 1]
raw_proba_test = lgbm_model.predict_proba(X_test_df)[:, 1]

print(f"Probabilidades RAW val  — min={raw_proba_val.min():.6f}, "
      f"max={raw_proba_val.max():.6f}, media={raw_proba_val.mean():.6f}")
print(f"Probabilidades RAW test — min={raw_proba_test.min():.6f}, "
      f"max={raw_proba_test.max():.6f}, media={raw_proba_test.mean():.6f}")
print("\nNota: calibración isotónica deshabilitada (comprimía los scores al rango "
      "de prevalencia real y rompía el F1).")

In [ ]:
# Celda 8 — Búsqueda de umbral óptimo y evaluación
#
# 1. Precision-Recall curve en val → umbral que maximiza F1.
# 2. Se aplica ese mismo umbral (fijado en val) a test. No re-optimizar en test.
#
# Criterios de éxito para modelo de TEMPORADA (mismos que anual pero más alcanzables):
#   AUC-PR > 0.10   — objetivo mínimo
#   F1     > 0.20   — objetivo mínimo

import numpy as np
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)

# Objetivos para modelo de temporada
AUC_PR_TARGET = 0.10
F1_TARGET = 0.20

# ── Paso 1: encontrar umbral óptimo en val ────────────────────────────────────
precisions, recalls, thresholds_pr = precision_recall_curve(y_val, raw_proba_val)
f1_scores = 2 * precisions * recalls / (precisions + recalls + 1e-9)
best_idx = int(f1_scores[:-1].argmax())
optimal_threshold = float(thresholds_pr[best_idx])
best_f1_val = float(f1_scores[best_idx])

print("Búsqueda de umbral óptimo en val (curva precision-recall):")
print(f"  Umbral óptimo    : {optimal_threshold:.6f}")
print(f"  F1 máximo en val : {best_f1_val:.4f}")
print(f"  Precisión        : {precisions[best_idx]:.4f}")
print(f"  Recall           : {recalls[best_idx]:.4f}")


# ── Paso 2: función de evaluación ─────────────────────────────────────────────
def evaluate_split(
    proba: np.ndarray,
    y: np.ndarray,
    split_name: str,
    threshold: float,
) -> dict:
    """Calcula métricas completas para un split dado un array de probabilidades."""
    pred = (proba >= threshold).astype(int)

    metrics = {
        "auc_roc":   float(roc_auc_score(y, proba)),
        "auc_pr":    float(average_precision_score(y, proba)),
        "f1":        float(f1_score(y, pred, zero_division=0)),
        "precision": float(precision_score(y, pred, zero_division=0)),
        "recall":    float(recall_score(y, pred, zero_division=0)),
        "threshold": threshold,
    }

    print(f"\n{'='*55}")
    print(f"  {split_name.upper()} | threshold={threshold:.6f}")
    print(f"{'='*55}")
    print(f"  AUC-ROC   : {metrics['auc_roc']:.4f}")
    print(f"  AUC-PR    : {metrics['auc_pr']:.4f}")
    print(f"  F1        : {metrics['f1']:.4f}")
    print(f"  Precision : {metrics['precision']:.4f}")
    print(f"  Recall    : {metrics['recall']:.4f}")
    return metrics


# ── Paso 3: evaluar con el umbral óptimo (fijado en val, aplicado a test) ────
metrics_dict = {}
metrics_dict["val"]  = evaluate_split(raw_proba_val,  y_val,  "VAL",  optimal_threshold)
metrics_dict["test"] = evaluate_split(raw_proba_test, y_test, "TEST", optimal_threshold)

# ── Paso 4: verificar criterios de éxito ──────────────────────────────────────
auc_pr_val = metrics_dict["val"]["auc_pr"]

print("\n" + "="*55)
print("  CRITERIOS DE ÉXITO — MODELO TEMPORADA")
print("="*55)
print(f"  {'[OK]' if auc_pr_val > AUC_PR_TARGET else '[!!]'} AUC-PR val > {AUC_PR_TARGET}: "
      f"{'PASA' if auc_pr_val > AUC_PR_TARGET else 'FALLA'}")
print(f"  {'[OK]' if best_f1_val > F1_TARGET else '[!!]'} F1 val (umbral óptimo) > {F1_TARGET}: "
      f"{'PASA' if best_f1_val > F1_TARGET else 'FALLA'}")
print(f"  {'[OK]' if lgbm_model.best_iteration_ > 1 else '[!!]'} best_iteration_ > 1: "
      f"{'PASA' if lgbm_model.best_iteration_ > 1 else 'FALLA'}")
print(f"  [INFO] Umbral óptimo encontrado: {optimal_threshold:.6f}")

all_pass = (auc_pr_val > AUC_PR_TARGET) and (best_f1_val > F1_TARGET) and (lgbm_model.best_iteration_ > 1)
print("="*55)
if all_pass:
    print("  >> MODELO TEMPORADA LISTO PARA GUARDAR <<")
else:
    print("  >> REVISA LOS CRITERIOS FALLIDOS ANTES DE REPORTAR <<")
print("="*55)

In [ ]:
# Celda 9 — Guardar modelo de temporada
#
# Artefacto en formato dict para facilitar inspección sin reimportar clases.
# threshold = umbral óptimo encontrado en val (no fijo).
# calibration = 'none' — se usan probabilidades RAW del LightGBM.
# training_mode = 'season_mar_may' — IMPORTANTE: solo usar en producción durante mar-may.

import joblib

artifact = {
    # Objeto del modelo
    "model": lgbm_model,
    # Configuración de inferencia
    "feature_names": available_features,
    "threshold": optimal_threshold,       # umbral aprendido de val, no fijo
    "calibration": "none",                # sin calibración isotónica
    # Metadata de modo de entrenamiento — CRÍTICO para inferencia
    "training_mode": "season_mar_may",    # modelo válido SOLO para meses 3, 4, 5
    "season_months": [3, 4, 5],           # meses de la temporada crítica
    "subsample_ratio": NEG_RATIO,         # ratio de subsampleo usado (5:1)
    # Métricas de evaluación
    "metrics": metrics_dict,
    # Metadatos del entrenamiento
    "random_seed": RANDOM_SEED,
    "best_iteration": lgbm_model.best_iteration_,
    "train_split": "2018-2022 (meses 3-5)",
    "val_split": "2023 (meses 3-5)",
    "test_split": "2024 (meses 3-5)",
}

model_path = f"{OUTPUT_DIR}/{MODEL_FILENAME}"
joblib.dump(artifact, model_path, compress=3)   # compress=3 reduce tamaño ~40%

print(f"Modelo guardado en: {model_path}")
print(f"Umbral guardado  : {optimal_threshold:.6f}")
print(f"Calibración      : none (probabilidades RAW)")
print(f"Modo de uso      : SOLO durante temporada {artifact['season_months']}")

# Mostrar tamaño del archivo
size_mb = os.path.getsize(model_path) / 1e6
print(f"Tamaño del artefacto: {size_mb:.1f} MB")

## Después de entrenar

> **IMPORTANTE:** Este modelo es el `m1_lightgbm_season.joblib` — el modelo de **temporada (mar-may)**.
> Solo debe usarse en producción durante los meses de marzo, abril y mayo.
> Para predicciones fuera de temporada, usar el modelo anual `m1_lightgbm_colab.joblib`.

### 1. Descarga el artefacto
Desde Google Drive, descarga **`m1_lightgbm_season.joblib`** (ruta: `petenfire/models/`).

### 2. Cópialo al proyecto local
```bash
cp ~/Downloads/m1_lightgbm_season.joblib <proyecto>/models_artifacts/m1/
```

### 3. Reporta las métricas al orquestador

Indica los siguientes valores de la celda 8:

| Métrica | Valor val | Valor test |
|---------|-----------|------------|
| AUC-PR  | _rellenar_ | _rellenar_ |
| AUC-ROC | _rellenar_ | _rellenar_ |
| F1 (umbral óptimo) | _rellenar_ | _rellenar_ |
| Umbral óptimo | _rellenar_ | — |
| best_iteration_ | _rellenar_ | — |

### 4. Si algún criterio falla
- **AUC-PR ≤ 0.10**: las features pueden tener demasiados NaN. Ejecuta la celda 5
  e inspecciona el diagnóstico de NaN al final de la celda.
- **F1 ≤ 0.20 (umbral óptimo)**: el modelo no discrimina lo suficiente. Prueba
  aumentar `N_ESTIMATORS` a 1000 o reducir `LEARNING_RATE` a 0.02.
- **best_iteration_ = 1**: el modelo no aprendió. Verifica que `y_train.sum() > 0`
  y que `X_train` no tenga todas las columnas en NaN.
- **Todos los scores se concentran en rango estrecho**: normal con base rate baja;
  el umbral óptimo encontrado en la curva PR ya lo compensa.

### 5. Inferencia con el artefacto guardado (SOLO temporada)
```python
import joblib
import pandas as pd

art = joblib.load("models_artifacts/m1/m1_lightgbm_season.joblib")
model     = art["model"]
threshold = art["threshold"]        # umbral óptimo aprendido de val
features  = art["feature_names"]
season    = art["season_months"]    # [3, 4, 5] — validar antes de inferir

# SIEMPRE verificar que estamos en temporada antes de usar este modelo
current_month = pd.Timestamp.now().month
if current_month not in season:
    raise ValueError(
        f"Mes {current_month} fuera de temporada {season}. "
        "Usar modelo anual 'm1_lightgbm_colab.joblib' en su lugar."
    )

X_new = pd.DataFrame(new_data, columns=features)
proba = model.predict_proba(X_new)[:, 1]
pred  = (proba >= threshold).astype(int)
```